# Suite DST — Distributions

A distribution of weights is over the matches of its symbols: its cases, each a weight and a predicate over the same
symbols, given by reference or inline, are listed in decreasing precedence, and a match weighs what the first case whose
predicate holds of it says. `Sample` draws a store's matches in proportion to their weights, from a random source the
caller gives.

In [ ]:
import { Expressions as E } from "@mbse/expressions";
import { Terms } from "@mbse/expressions/Framework";
import { Constraints as C, Distributions as D, Predicates as P } from "@mbse/patterns";
import { Errors, JSON as SchemaJSON, Schemas as S, Stores } from "@mbse/schemas/Framework";
import { Contact, assert, book, listed, raises, same } from "./support.js";

const APerson = { person: Contact };
const HasName = new P.OfPredicate.Builder().name("HasName").symbols(APerson).parameters((p) => p.name("name"))
  .requires(E.variable("person").name.eq(E.variable("name"))).create();
const Names = new D.OfWeights.Builder().name("Names").symbols(APerson).decreasing(
  (wt) => wt.weight(10).requires((pred) => pred.symbols(APerson).requires(HasName.call(pred.person, "alice"))),
  (wt) => wt.weight(5).requires((pred) => pred.symbols(APerson).requires(HasName.call(pred.person, "ben"))),
  (wt) => wt.weight(15).requires((pred) => pred.symbols(APerson).requires(HasName.call(pred.person, "chermon"))),
  (wt) => wt.weight(7).requires((pred) => pred.symbols(APerson).requires(HasName.call(pred.person, "davi"))),
).create();

function counted(names: string[]): Map<string, number> {
  const counts = new Map<string, number>();
  for (const name of names) counts.set(name, (counts.get(name) ?? 0) + 1);
  return counts;
}

function take<T>(items: Iterator<T>, n: number): T[] {
  return Array.from({ length: n }, () => items.next().value as T);
}

## DST-01 · A distribution is built fluently: symbols, and cases in decreasing precedence

In [ ]:
{
  assert(Names.name === "Names" && same(Names.symbols.keys(), ["person"]) && Names.symbols.get("person") === Contact);
  assert(same(Names.cases.map((c) => c.weight), [10, 5, 15, 7])); // in the order given
  assert(Names.cases.every((c) => c.predicate.rule.predicate === HasName)); // one predicate, by reference
  assert(Names.cases[0].predicate.name === null && Names.cases[0].predicate.rule.arguments[1].value === "alice"); // inline
  const Adult = new P.OfPredicate.Builder().name("Adult").symbols(APerson).requires(E.variable("person").age.ge(18n)).create();
  const more = new D.OfWeights.Builder(Names).decreasing((wt) => wt.weight(1).requires(Adult)).clone(); // by reference too
  assert(more.cases.length === 5 && more.cases[4].predicate === Adult && Names.cases.length === 4);
  assert(same(more.symbols.values(), Names.symbols.values()) && more.symbols !== Names.symbols);
  assert(new D.OfWeights.Builder(more).name("More").update() === more && more.name === "More");
  const builder = new D.OfWeights.Builder().symbols(APerson);
  assert(Terms.same(builder.person.data, E.variable("person").data)); // the builder gives its symbols as variables
  const c = D.OfCase.resolve((wt: D.OfCase.Builder) => wt.weight(2).requires(Adult));
  assert(c.weight === 2 && c.predicate === Adult && D.OfCase.resolve(c) === c);
  raises(TypeError, () => D.OfCase.resolve("x"), "expected a case or a callable taking its builder, got 'x'");
  assert(D.DIALECT.base === P.DIALECT && same([...D.DIALECT.kinds().keys()].slice(-2), ["weights", "case"]));
  assert(Names.schema_name() === "Patterns.OfWeights" && Names.cases[0].schema_name() === "Patterns.OfCase");
}

## DST-02 · Distributions are checked when asked, and when drawn from

In [ ]:
{
  assert(D.DIALECT.validate(Names, { core: true }).length === 0);
  const Value = new S.OfObject.Builder().name("Value").create();
  const odd = new D.OfWeights.Builder().symbols({ person: Contact, v: Value }).decreasing(
    (wt) => wt.weight(-1).requires(HasName),
    (wt) => wt.weight(1).requires((pred) => pred.symbols({ c: Contact }).requires(true))).create();
  assert(same(D.DIALECT.validate(odd), [
    "symbol 'v' needs a named reference object schema",
    "case 0: its predicate's symbols must be the distribution's", "case 0: its predicate has parameters; apply it instead",
    "case 1: its predicate's symbols must be the distribution's", "argument 0: a case's weight must be positive, got -1.0"]));
  assert(same(D.DIALECT.validate(new D.OfWeights.Builder().symbols(APerson).create()), ["a distribution needs a case"]));
  assert(same(D.DIALECT.validate(new D.OfWeights.Builder().symbols(new Map([["person", "Contact"]])).decreasing((wt) => wt.weight(1).requires(
    (pred) => pred.symbols(APerson).requires(true))).create()), [
    "symbol 'person' needs a named reference object schema", "case 0: its predicate's symbols must be the distribution's"]));
  assert(same(D.DIALECT.validate(new D.OfWeights(null, [E.literal(1n).data], new Map(Object.entries(APerson)))),
    ["a distribution's arguments are cases"]));
  assert(same(D.DIALECT.validate(new D.OfCase(1, E.literal(true).data)), ["a case's predicate must be a predicate"]));
  assert(same(D.DIALECT.validate(new D.OfCase(1, null)), ["a case needs a predicate"]));
  assert(same(D.DIALECT.validate(new D.OfWeights(null, [new D.OfCase(1, null)], new Map(Object.entries(APerson)))),
    ["argument 0: a case needs a predicate"]));
  raises(Errors.ValueError, () => D.Sample(book(), new D.OfWeights.Builder().symbols(APerson).create(), new Stores.PCG32(42n)),
    "the distribution cannot be drawn from: a distribution needs a case");
  assert(D.check(Names) === Names);
}

## DST-03 · A match weighs what its first holding case says

In [ ]:
{
  const store = book();
  const [alice, ben, nobody] = ["alice", "ben", "zed"].map((n) => store.Contact().name(n).create());
  const evaluate = new P.Evaluator(store);
  assert(same([alice, ben, nobody].map((p) => D.weight(evaluate, Names, { person: p })), [10, 5, 0]));
  const Overlapping = new D.OfWeights.Builder().symbols(APerson).decreasing(
    (wt) => wt.weight(3).requires((pred) => pred.symbols(APerson).requires(pred.person.has("name"))),
    (wt) => wt.weight(2).requires((pred) => pred.symbols(APerson).requires(HasName.call(pred.person, "alice")))).create();
  assert(D.weight(evaluate, Overlapping, { person: alice }) === 3); // the first case that holds, not the most specific
  assert(D.weight(evaluate, Names, { person: store.Contact().create() }) === 0); // unknown is not holding
}

## DST-04 · Sample draws the store's matches in proportion to their weights

In [ ]:
{
  const store = book();
  const people = ["alice", "ben", "chermon", "davi", "zed"].map((n) => store.Contact().name(n).create());
  listed(store, ...people);
  const drawn = D.Sample(store, Names, new Stores.PCG32(42n));
  const names = counted(take(drawn, 3700).map((m: any) => m.person.name));
  assert(same([...names.keys()].sort(), ["alice", "ben", "chermon", "davi"])); // zed weighs nothing
  for (const [n, w] of [["alice", 10], ["ben", 5], ["chermon", 15], ["davi", 7]] as const) {
    assert(Math.abs((names.get(n) as number) - 100 * w) < 100, n);
  }
  const again = [0, 1, 2].map(() => (D.Sample(store, Names, new Stores.PCG32(42n)).next().value as any).person.name);
  assert(same(again, [again[0], again[0], again[0]])); // the same seed, the same draws
  const first = take(D.Sample(store, Names, new Stores.PCG32(42n)), 6).map((m: any) => m.person.name);
  const other = take(D.Sample(store, Names, new Stores.PCG32(42n).split("other")), 6).map((m: any) => m.person.name);
  assert(!same(first, other)); // another stream, split from the same seed
  const nobody = book();
  listed(nobody, nobody.Contact().name("zed").create());
  raises(Errors.ValueError, () => D.Sample(nobody, Names, new Stores.PCG32(1n)).next(), "no match of the store's data has a weight");
}

## DST-05 · Distributions are data: a predicate used by several cases is written once

In [ ]:
{
  const store = new C.OfStore(book());
  const text = SchemaJSON.ToJSON(store).Reachable(D.OfWeights.Schema, Names);
  assert(text.split('"name": "HasName"').length - 1 === 1 && text.split('"kind": "predicate"').length - 1 === 5); // HasName, and four inline
  assert(SchemaJSON.ToJSON(C.Builders).Reachable(D.OfWeights.Schema, Names) === text);
  const copy = SchemaJSON.FromJSON(store).Reachable(D.OfWeights.Schema, text) as D.OfWeights;
  assert(copy.symbols.get("person") === Contact && same(copy.cases.map((c) => c.weight), [10, 5, 15, 7]));
  assert(new Set(copy.cases.map((c) => c.predicate.rule.predicate)).size === 1); // still one predicate
  assert(copy.cases.every((c, i) => Terms.same(c.predicate.rule, Names.cases[i].predicate.rule)));
  assert(D.DIALECT.validate(copy, { core: true }).length === 0);
}